## Data loads

In [ ]:
import pandas as pd
import os
import re
from glob import glob

from src.utils.preprocessing import Preprocessor

from sklearn.metrics import confusion_matrix, f1_score, accuracy_score
from matplotlib import pyplot as plt
from matplotlib import font_manager as fm
import seaborn as sbn

from collections import Counter
from tqdm import tqdm
import numpy as np


font_fname = os.environ.get('PROJECT_FONT_PATH', '')  # point at any local CJK/serif .ttf
font_family = fm.FontProperties(fname=font_fname).get_name() #폰트 설정
plt.rcParams["font.family"] = font_family  #폰트 적용



def check_label(df, s):
    df = df.to_frame()
    df['label'] = s
    return df


negative_patterns = ['without','not? (sugges|compat)', 'unlike', 'neither', 'no', 'negative']
uncertain_patterns = ['rule out', 'Rule Out', "Rule out"]
measure_dict = dict()

In [ ]:
path = os.path.join('data','reports','*')
glob(path)

In [ ]:
df = pd.read_csv(glob(path)[0], encoding = 'utf-8-sig', engine = 'c').reset_index()

text_df = df['검사결과결론내용'].dropna()

preprocessor = Preprocessor(
    df = text_df,
    negative_patterns = negative_patterns,
    uncertain_patterns = uncertain_patterns
)


In [ ]:
temp_df = df[['수술일자']].set_axis(['date'], axis = 1)

temp_df.loc[:, 'recur'] = df['검사결과결론내용'].apply(lambda x: bool(re.search(r'recur', x.lower())))
temp_df.loc[:, 'metas'] = df['검사결과결론내용'].apply(lambda x: bool(re.search(r'metas', x.lower())))

In [ ]:
temp_df.recur.sum(), temp_df.metas.sum()

In [ ]:
target = 'recur'


target_name = 'Metastasis' if 'metas' in target else 'Recurrence'
# 'date' 컬럼을 datetime 포맷으로 변환
temp_df['date'] = pd.to_datetime(temp_df['date'], format='%Y%m%d')

# 3개월(분기) 단위로 그룹화하여 flag 카운트 집계
temp_df['quarter'] = temp_df['date'].dt.to_period('Q')
quarter_group = temp_df.groupby(['quarter', target]).size().unstack()
quarter_group.index = quarter_group.index.astype(str)

# 시각화
# plt.figure(figsize=(12,6))
plt.rcParams['font.size'] = 12

fig, ax = plt.subplots(1, 1, figsize = (12, 3))
quarter_group.plot(kind='bar', stacked=True, ax=ax, color = ['#7979C9','grey'])

# Set x-axis with multi-level ticks: outer for year, inner for month (or quarter)
# First, get year and quarter labels
quarters = pd.PeriodIndex(quarter_group.index, freq='Q')
years = [str(q.year) for q in quarters]
qs = ['Q' + str(q.quarter) for q in quarters]

# Set inner xticks as quarters
ax.set_xticks(range(len(quarter_group.index)))
ax.set_xticklabels(qs, rotation=0)

# Add outer ticks for years
# Find indices where year changes
from matplotlib.ticker import FixedLocator, FixedFormatter

year_change_indices = [i for i in range(len(years)) if (i == 0 or years[i] != years[i-1])]
year_labels = [years[i] for i in year_change_indices]
year_positions = []

for i in range(len(year_change_indices)):
    start = year_change_indices[i]
    end = year_change_indices[i+1] if i+1 < len(year_change_indices) else len(quarters)
    center = (start + end - 1) / 2
    year_positions.append(center)

ax2 = ax.twiny()
ax2.set_xticks(year_positions)
ax2.set_xticklabels(year_labels, fontweight='bold')
ax2.set_xlim(ax.get_xlim())
ax2.tick_params(axis='x', length=0, pad=25)
ax2.xaxis.set_ticks_position('bottom')  # Hide top and bottom ticks

ax.set_xlabel('')
ax2.set_xlabel('Quarter')
ax2.xaxis.set_label_position('bottom')
ax.set_ylabel('Frequency')

tax = ax.twinx()
tax.plot(quarter_group.apply(lambda x: x.values[1] / x.values[0] * 100, axis = 1), marker = 'o', color = 'tab:red')
tax.set_ylabel('Unlabeled rate (%)', rotation = 270, labelpad = 20)

ax.set_title(f'Unlabeled {target_name.lower()} rate by quarter')

# ax에 그려진 정보들을 가져와서 legend용 handle/label 직접 구성
bar_handles = [c for c in ax.containers]
bar_labels = ['Presumed negative', 'Unlabeled recurrence']
line_handles = tax.lines
line_labels = ['Unlabeled/Presumed']

ax.legend(
    bar_handles + line_handles, bar_labels + line_labels,
    loc = 'lower left', 
    framealpha = .9
    )


fig.tight_layout()
fig.show()
fig.savefig(os.path.join(f'{target_name}_quarter_unlabeled_rate.png'), dpi = 400)

In [ ]:
target = 'metas'


target_name = 'Metastasis' if 'metas' in target else 'Recurrence'
# 'date' 컬럼을 datetime 포맷으로 변환
temp_df['date'] = pd.to_datetime(temp_df['date'], format='%Y%m%d')

# 3개월(분기) 단위로 그룹화하여 flag 카운트 집계
temp_df['quarter'] = temp_df['date'].dt.to_period('Q')
quarter_group = temp_df.groupby(['quarter', target]).size().unstack()
quarter_group.index = quarter_group.index.astype(str)

# 시각화
# plt.figure(figsize=(12,6))
plt.rcParams['font.size'] = 12

fig, ax = plt.subplots(1, 1, figsize = (12, 3))
quarter_group.plot(kind='bar', stacked=True, ax=ax, color = ['#7979C9','grey'])

# Set x-axis with multi-level ticks: outer for year, inner for month (or quarter)
# First, get year and quarter labels
quarters = pd.PeriodIndex(quarter_group.index, freq='Q')
years = [str(q.year) for q in quarters]
qs = ['Q' + str(q.quarter) for q in quarters]

# Set inner xticks as quarters
ax.set_xticks(range(len(quarter_group.index)))
ax.set_xticklabels(qs, rotation=0)

# Add outer ticks for years
# Find indices where year changes
from matplotlib.ticker import FixedLocator, FixedFormatter

year_change_indices = [i for i in range(len(years)) if (i == 0 or years[i] != years[i-1])]
year_labels = [years[i] for i in year_change_indices]
year_positions = []

for i in range(len(year_change_indices)):
    start = year_change_indices[i]
    end = year_change_indices[i+1] if i+1 < len(year_change_indices) else len(quarters)
    center = (start + end - 1) / 2
    year_positions.append(center)

ax2 = ax.twiny()
ax2.set_xticks(year_positions)
ax2.set_xticklabels(year_labels, fontweight='bold')
ax2.set_xlim(ax.get_xlim())
ax2.tick_params(axis='x', length=0, pad=25)
ax2.xaxis.set_ticks_position('bottom')  # Hide top and bottom ticks

ax.set_xlabel('')
ax2.set_xlabel('Quarter')
ax2.xaxis.set_label_position('bottom')
ax.set_ylabel('Frequency')

tax = ax.twinx()
tax.plot(quarter_group.apply(lambda x: x.values[1] / x.values[0] * 100, axis = 1), marker = 'o', color = 'tab:red')
tax.set_ylabel('Unlabeled rate (%)', rotation = 270, labelpad = 20)

ax.set_title(f'Unlabeled {target_name.lower()} rate by quarter')

# ax에 그려진 정보들을 가져와서 legend용 handle/label 직접 구성
bar_handles = [c for c in ax.containers]
bar_labels = ['Presumed negative', 'Unlabeled recurrence']
line_handles = tax.lines
line_labels = ['Unlabeled / Presumed']

ax.legend(
    bar_handles + line_handles, bar_labels + line_labels,
    loc = 'best', 
    framealpha = .9
    )
ax.legend().remove()


fig.tight_layout()
fig.show()
fig.savefig(os.path.join(f'{target_name}_quarter_unlabeled_rate.png'), dpi = 400)

In [ ]:


def plot_quarter_unlabeled_rate(df, target, ax, xticks = True):
    target_name = 'Metastasis' if 'metas' in target else 'Recurrence'
    # 'date' 컬럼을 datetime 포맷으로 변환
    temp_df['date'] = pd.to_datetime(temp_df['date'], format='%Y%m%d')

    # 3개월(분기) 단위로 그룹화하여 flag 카운트 집계
    temp_df['quarter'] = temp_df['date'].dt.to_period('Q')
    quarter_group = temp_df.groupby(['quarter', target]).size().unstack()
    quarter_group.index = quarter_group.index.astype(str)

    quarter_group.plot(kind='bar', stacked=True, ax=ax, color = ['#7979C9','grey'])

    # Set x-axis with multi-level ticks: outer for year, inner for month (or quarter)
    # First, get year and quarter labels
    quarters = pd.PeriodIndex(quarter_group.index, freq='Q')
    years = [str(q.year) for q in quarters]
    qs = ['Q' + str(q.quarter) for q in quarters]

    # Add outer ticks for years
    # Find indices where year changes
    from matplotlib.ticker import FixedLocator, FixedFormatter

    year_change_indices = [i for i in range(len(years)) if (i == 0 or years[i] != years[i-1])]
    year_labels = [years[i] for i in year_change_indices]
    year_positions = []

    for i in range(len(year_change_indices)):
        start = year_change_indices[i]
        end = year_change_indices[i+1] if i+1 < len(year_change_indices) else len(quarters)
        center = (start + end - 1) / 2
        year_positions.append(center)


    if xticks:
        # Set inner xticks as quarters
        ax.set_xticks(range(len(quarter_group.index)))
        ax.set_xticklabels(qs, rotation=0)

        ax2 = ax.twiny()
        ax2.set_xticks(year_positions)
        ax2.set_xticklabels(year_labels, fontweight='bold')
        ax2.set_xlim(ax.get_xlim())
        ax2.tick_params(axis='x', length=0, pad=25)
        ax2.xaxis.set_ticks_position('bottom')  # Hide top and bottom ticks

        ax.set_xlabel('')
        ax2.set_xlabel('Quarter')
        ax2.xaxis.set_label_position('bottom')

    ax.set_ylabel(f'{target_name} frequency')

    tax = ax.twinx()
    tax.plot(quarter_group.apply(lambda x: x.values[1] / x.values[0] * 100, axis = 1), marker = 'o', color = 'tab:red')
    tax.set_ylabel('Unlabeled rate (%)', rotation = 270, labelpad = 20)

    # ax.set_title(f'Unlabeled {target_name.lower()} rate by quarter')

    # ax에 그려진 정보들을 가져와서 legend용 handle/label 직접 구성
    bar_handles = [c for c in ax.containers]
    bar_labels = ['Presumed negative', 'Unlabeled recurrence']
    line_handles = tax.lines
    line_labels = ['Unlabeled/Presumed']
    ax.legend().remove()
    return ax, bar_handles + line_handles, bar_labels + line_labels

target = 'recur'

# 시각화
# plt.figure(figsize=(12,6))
plt.rcParams['font.size'] = 12

fig, ax = plt.subplots(2, 1, figsize = (12, 5), sharex = True)

ax[0], *_ = plot_quarter_unlabeled_rate(df, 'recur', ax[0], False)
ax[1], handles, labels = plot_quarter_unlabeled_rate(df, 'metas', ax[1])

# Add extra vertical space at the top for the legend
fig.tight_layout(rect=[0, 0, 1, 0.94])  # Adjust rect's top to leave space

fig.legend(
    handles, labels,
    loc = 'upper center', 
    framealpha = .9,
    ncols = 3,
    bbox_to_anchor=(0.5, 0.98)  # Move legend slightly up, but within left/right bounds
)

fig.show()
fig.savefig(os.path.join(f'Quarter_unlabeled_rate.png'), dpi = 400)